# Vision Transformers

#Steps:
Patch extraction

1.   Patch embedding
2. CLS token
3. Positional embeddings
4. Single-head attention first
5. Multi-head attention
6. MLP block
7. One Transformer encoder block
8. Stack 12 encoder blocks
9. Classification head
10. Training loop
11. Compare with 97% CNN


##1. patch extarction

*   load the image
*   make the patches for every image
the size of the image patches like that
[channels,patch rows,patch rows, patch columns, patch height , patch width]
*   flatten the patches so they are like the transformers tokens







#2. CLS token
CLS token as a learned summary representation of the whole image
it has the same size as the other tokens the features that will be in it will the most important ones that we get from the other tokens so . it contains information about all the other tokens

##3. positional embedding:
in here the positional embeddings are the position informatin that the transformer wants to know the position of each token the original transformer they provided as give not learnable but in vit they thought that lets make it a learnable as maybe the model think this representation are not good for the model

## 4. Single-head attention
the idea behind it is that for all the patches i have to make their q ,k and value vector but at this point we need to just make it as learnable parameter

define a class for vision transformer

In [70]:
from google.colab import files
files.upload()
!unzip "leaves (1).zip" -d dataset

KeyboardInterrupt: 

In [126]:
from torch import nn
class VisionTransformer(nn.Module):
  def __init__(self,patch_size = 16,channels = 3,embed_dim = 512, image_size=224,head_nums= 8):
    super().__init__()
    self.patch_size = patch_size
    self.patch_dim = channels * patch_size * patch_size
    self.patch_nums= (image_size//patch_size)**2
    self.head_nums = head_nums
    self.embed_dim = embed_dim

    self.patch_embeddings= nn.Linear(self.patch_dim , self.embed_dim )
    self.cls = nn.Parameter(torch.randn(1,1,self.embed_dim))
    self.POS = nn.Parameter(torch.randn(1,self.patch_nums+1,self.embed_dim))
    self.WQ = nn.Linear(self.embed_dim,self.embed_dim)
    self.WK = nn.Linear(self.embed_dim,self.embed_dim)
    self.WV = nn.Linear(self.embed_dim,self.embed_dim)

  def make_patches(self,images):
    """ take the image and patch it pased on the patch size and return the flatten patches """
    print(images.shape)
    B, C, H, W = images.shape
    patches = images.unfold(2, self.patch_size, self.patch_size).unfold(3, self.patch_size, self.patch_size)
    patches=patches.permute(0,2,3,1,4,5)
    patches = patches.reshape(B, self.patch_nums,-1)
    return patches

  def foreward(self,x):
    patches = self.make_patches(x)
    B,num_of_patches, num_of_features = patches.shape


    patches_embbedding = self.patch_embeddings(patches)
    print(patches_embbedding.shape)
    cls = self.cls.expand(B,-1,-1)
    all_patches = torch.cat((cls,patches_embbedding),dim=1)
    pos=self.POS.expand(B,-1,-1)
    all_patches = all_patches + pos
    Q = self.WQ(all_patches)
    K = self.WK(all_patches)
    V = self.WV(all_patches)
    # Multi-head attention splits the Q, K, and V feature dimensions into multiple heads
    # so that the model can learn multiple attention relationships between the patches in parallel.
    Q =Q.reshape(B,num_of_patches+1,self.head_nums,-1)
    K =K.reshape(B,num_of_patches+1,self.head_nums,-1)
    V =V.reshape(B,num_of_patches+1,self.head_nums,-1)
    # i stopped here i need to check and reshape
    print(Q.shape)
    print(K.shape)
    print(V.shape)
    # print(Q.shape)
    # return Q,K,V







In [127]:
from PIL import Image
from torchvision.datasets import ImageFolder
from torchvision import transforms, torch
from torch.utils.data import random_split
from torch.utils.data import DataLoader
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
dataset = ImageFolder(root="dataset/images", transform=transform)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])



In [128]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [129]:
images,labels = next(iter(train_loader))
print(images.shape)

torch.Size([32, 3, 224, 224])


In [130]:
model = VisionTransformer(patch_size=16)
model.foreward(images)

torch.Size([32, 3, 224, 224])
torch.Size([32, 196, 512])
torch.Size([32, 197, 8, 64])
torch.Size([32, 197, 8, 64])
torch.Size([32, 197, 8, 64])


In [23]:
for batch in train_loader:
  print(batch[0].shape)

torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size([32, 3, 224, 224])
torch.Size